# Knowledge Base Creation

* Create a knowledge base for the Credit Card FAQ chatbot to retrieve information from for its response
* All information is publicly available on the CBA website, **no** internal data involved

### 0. Install and Import packages

In [1]:
# !pip install --upgrade pip
# !conda install -c anaconda requests -y
# !pip install pandas
# !pip install selenium
# !pip install chromadb sentence-transformers
# !pip install openai
# !pip install bs4
# !pip install lxml

In [1]:
import sys
import time
import os 
import tempfile
from dotenv import load_dotenv 

import json
import pandas as pd
from collections import Counter
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

import openai
import tiktoken
import chromadb

pd.set_option("display.max_rows", None)

### 1. Scrape CC-related Links on the Commbank Website

1a. Scrape the sitemap for static HTML links

In [2]:
# Fetch the sitemap
sitemap = 'https://www.commbank.com.au/sitemap.xml'
response = requests.get(sitemap)
soup = BeautifulSoup(response.content, 'xml')

# Find all <loc> tags and filiter URLs containing 'credit-card'
urls = [loc.text for loc in soup.find_all('loc') 
        if 'credit-card' in loc.text 
            and 'news' not in loc.text
            and 'eform' not in loc.text
            and 'tools' not in loc.text] # Excludinging urls containing 'news'/ 'eform'/ 'tools'

# Output the urls
url_list = []
for url in urls:
    url_list.append(url)

print(len(url_list))

84


In [3]:
# Save the output to /data
with open("../data/cc_urls.txt", "w") as f:
    for url in url_list:
        f.write(url + "\n")

print("Saved to ../data/cc_urls.txt")

92

79

85

92

86

...

85

78

72

78

Saved to ../data/cc_urls.txt


1b. Scrape dynamically rendered content on the 'Manage' page

In [5]:
# Set up Selenium WebDriver 
options = webdriver.ChromeOptions()
options.add_argument("--headless") #Run without opening a browser window
driver = webdriver.Chrome(options=options)

mng_url = "https://www.commbank.com.au/credit-cards/manage.html"
driver.get(mng_url)

# Extract <a> tags using Selenium
slnm_links = [a.get_attribute("href") for a in driver.find_elements(By.TAG_NAME, "a") if a.get_attribute("href")]

# Extract <a> tags using BeautifulSoup (static HTML)
soup = BeautifulSoup(driver.page_source, "html.parser")
bs4_links = [a["href"] for a in soup.find_all("a", href=True)]

# Identify dynamically generated links 
dyno_links = set(slnm_links) - set(bs4_links)
dyno_urls = sorted(dyno_links) # Turn into a sorted list

# Save dyno links to txt
with open("../data/dyno_urls.txt", "w") as f:
    for url in dyno_urls:
        f.write(str(url) + "\n")

driver.quit()

63

83

71

75

134

...

72

86

76

55

78

In [6]:
# Remove irrelevant links 
dyno_urls = [url for url in dyno_urls 
             if '/about-us' not in str(url)
                and '/contact-us' not in str(url)
                and '/travel' not in str(url)
                and '.pdf' not in str(url)
                and 'footer' not in str(url)
                and 'html#' not in str(url)]

# Save dyno links to txt
with open("../data/dyno_urls.txt", "w") as f:
    for url in dyno_urls:
        f.write(str(url) + "\n")

92

88

110

98

87

...

115

54

81

95

86

Combine the two lists and dedup

In [11]:
# Merge the two lists
merge_list = url_list + dyno_urls
merge_list_dedup = sorted(set(merge_list))

print(len(merge_list_dedup))

104


In [12]:
print(merge_list_dedup)

['https://www.commbank.com.au/articles/credit-cards/common-credit-card-mistakes-to-avoid.html', 'https://www.commbank.com.au/articles/credit-cards/credit-card-definitions.html', 'https://www.commbank.com.au/articles/credit-cards/credit-card-minimum-repayment.html', 'https://www.commbank.com.au/articles/credit-cards/how-are-credit-card-payments-applied.html', 'https://www.commbank.com.au/articles/credit-cards/how-do-credit-card-points-work.html', 'https://www.commbank.com.au/articles/credit-cards/how-do-credit-cards-work.html', 'https://www.commbank.com.au/articles/credit-cards/how-does-credit-card-interest-work.html', 'https://www.commbank.com.au/articles/credit-cards/how-to-choose-your-first-credit-card.html', 'https://www.commbank.com.au/articles/credit-cards/how-to-get-a-credit-card.html', 'https://www.commbank.com.au/articles/credit-cards/how-to-minimise-credit-card-fees.html', 'https://www.commbank.com.au/articles/credit-cards/how-to-understand-and-check-your-credit-score.html', '

In [8]:
# Manual clean up - outdated link/ manual dedup

rmv = ['https://www.commbank.com.au/articles/credit-cards/what-makes-up-your-credit-card-balance.html?ei=paying_repay'
    , 'https://www.commbank.com.au/business/business-credit-cards/business-awards-credit-cards.html'
    , 'https://www.commbank.com.au/business/business-credit-cards/compare.html'
    , 'https://www.commbank.com.au/business/business-credit-cards/card-selector.html'
    , 'https://www.commbank.com.au/commbank-yello.html'
    , 'https://www.commbank.com.au/credit-cards.html'
    , 'https://www.commbank.com.au/business/business-credit-cards.html'
    , 'https://www.commbank.com.au/credit-cards/credit-card-offers.html'
    , 'https://www.commbank.com.au/credit-cards/credit-cards-offers/expired-offers.html'
    , 'https://www.commbank.com.au/credit-cards/diamond.html'
    , 'https://www.commbank.com.au/credit-cards/low-fee-gold.html'
    , 'https://www.commbank.com.au/credit-cards/low-rate-gold.html'
    , 'https://www.commbank.com.au/credit-cards/commbank-essentials.html'
    , 'https://www.commbank.com.au/credit-cards/manage/switch-card.html?ei=switch'
    , 'https://www.commbank.com.au/credit-cards/flightcentre.html'
    , 'https://www.commbank.com.au/credit-cards/myer.html'
    , 'https://www.commbank.com.au/credit-cards/platinum.html'
    , 'https://www.commbank.com.au/digital-banking/lock-block-limit-your-credit-card.html?ei=control_lbl'
    , 'https://www.commbank.com.au/digital-banking/lock-block-limit-your-credit-card.html'
    , 'https://www.commbank.com.au/support.html?ei=help_faqs']

master_list = [url for url in merge_list_dedup if url not in rmv]
print(len(master_list))

84


In [13]:
# Save the master list in  a text file 
with open("../data/master_list.txt", "w") as f:
    for url in master_list:
        f.write(str(url) + "\n")

92

79

85

92

86

...

115

81

78

95

86

### 2. Scrape the content on links in the master list

In [14]:
def scrape_pages(urls):
    """Scrapes dynamically rendered webpages using Safari WebDriver and stores raw content."""
    
    from selenium import webdriver
    from selenium.webdriver.safari.service import Service
    from bs4 import BeautifulSoup
    import time

    service = Service()  # Create a Safari WebDriver service instance
    driver = webdriver.Safari(service=service)  # Start Safari driver

    scraped_data = []  # Store data for all pages
    
    try:
        for url in urls:
            driver.get(url)  # Load the webpage
            time.sleep(3)  # Allow time for JavaScript to render content
            
            # Get HTML and parse it
            soup = BeautifulSoup(driver.page_source, "html.parser")

            # Extract page title (fallback if <title> tag is missing)
            title = soup.title.text.strip() if soup.title else "Untitled Page"

            # Remove unwanted elements
            for tag in soup(["script", "style", "header", "footer", "nav", "aside"]):
                tag.extract()

            # Remove specific <div> elements by class name
            unwanted_classes = ["skip-links-module", "commbank-header", "article-related list parbase", "experiencefragment"]
            for div in soup.find_all("div", class_=unwanted_classes):
                div.extract()

            # Extract cleaned text
            text = soup.get_text(separator="\n")
            text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])  # Remove empty lines
            
            # Store data in a structured format
            scraped_data.append({"url": url, "title": title, "content": text})

    except Exception as e:
        print(f"Error scraping {url}: {e}")

    finally:
        driver.quit()  # Close Safari browser

    return scraped_data

In [15]:
# Test out the function 
urls = master_list[:2]
scrape_pages(urls)

[{'url': 'https://www.commbank.com.au/articles/credit-cards/common-credit-card-mistakes-to-avoid.html',
  'title': 'Common credit card mistakes to avoid',
  'content': 'Common credit card mistakes to avoid\nCommon credit card mistakes to avoid\nHaving a good idea of how you’ll use your credit card can help you choose one that best suits your needs and offers the features and benefits you want.\nMaking a mistake can sometimes be a good way to learn new things. But when it comes to credit cards, a little bit of knowledge can help you make the most of your card without having to learn from your errors. Here are some common mistakes around credit cards to avoid.\n1. Choosing the wrong credit card\nThere are generally four different types of credit cards:\ninterest-free,\xa0low rate, low fee and awards\n. Each has its own benefits, and depending on how you want to use your credit card, one type may suit you better than the other. Take the time to\ncompare credit cards\nand make sure you pic

In [16]:
# Loop through all webpages in the master list 
urls = master_list
scraped_data = scrape_pages(urls)

# Check number of links scraped:
print(len(scraped_data))

84


### 3. Save Scraped Data 

In [17]:
# save scrapted data to a json file
with open("../data/scraped_data.json", "w", encoding='utf-8') as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)

### 4. Add Product Segment Tagging

In [9]:
# Reopen file for segment tagging 

import json
with open('../data/scraped_data.json', 'r', encoding='utf-8') as file:
    scraped_data = json.load(file)

In [10]:
# Tag each URL based on the product segment it's related to for later use

from collections import Counter

EXACT_GENERIC_URLS = {
    "https://www.commbank.com.au/credit-cards/awards-program.html",
    "https://www.commbank.com.au/credit-cards/qantas-frequent-flyer.html",
}

def infer_segment(url: str) -> str:
    # 1) exact exceptions
    if url in EXACT_GENERIC_URLS:
        return "generic"
    # 2) override rule
    if "/manage" in url:
        return "generic"
    if "/corporate" in url:
        return "corporate"
    # 3) prefix rules
    if url.startswith("https://www.commbank.com.au/business"):
        return "business"
    if url.startswith("https://www.commbank.com.au/credit-cards"):
        return "personal"
    # 4) everything else
    return "generic"

# Apply
for item in scraped_data:
    item["segment"] = infer_segment(item["url"])

# Check segment fallout
Counter([x["segment"] for x in scraped_data])


Counter({'generic': 50, 'personal': 20, 'business': 11, 'corporate': 3})

In [11]:
# Sanity Check

# Confirm manage pages are generic
manage_not_generic = [x["url"] for x in scraped_data if "/manage/" in x["url"] and x["segment"] != "generic"]
print("manage_not_generic:", manage_not_generic[:5])

# Confirm prefixes
weird_business = [x["url"] for x in scraped_data if x["url"].startswith("https://www.commbank.com.au/business") and x["segment"] != "business"]
print("weird_business:", weird_business[:5])


manage_not_generic: []
weird_business: ['https://www.commbank.com.au/business/business-credit-cards/corporate-charge-card.html', 'https://www.commbank.com.au/business/business-credit-cards/corporate-interest-free-days-card.html', 'https://www.commbank.com.au/business/business-credit-cards/corporate-low-rate-card.html']


In [12]:
# save scrapted data to a json file
with open("../data/scraped_data_w_segment.json", "w", encoding='utf-8') as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)